In [1]:
%load_ext autoreload
%autoreload 2

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import sys
import time
sys.path.append("../../../")
import src.experiments.textgenutils as gutils
from src.models.gpt import GPT, GPT2BPETokenizer
import transformers

In [3]:
# model_hf = transformers.GPT2LMHeadModel.from_pretrained("gpt2").cuda()
# config = {
#     'n_layers': len(model_hf.transformer.h),
#     'n_heads': model_hf.transformer.h[0].attn.num_heads,
#     'embed_dim': model_hf.transformer.h[0].attn.embed_dim,
#     'vocab_size': model_hf.lm_head.out_features,
#     'block_size': model_hf.transformer.wpe.num_embeddings,
#     'dropout_p': model_hf.transformer.drop.p
# }

In [8]:
model = GPT.from_pretrained("gpt2")
# model_right = gpt_right.GPTRight.from_pretrained("gpt2").cuda()

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 11526.45it/s]


In [10]:
def profile_kv_cache_generation_time_and_memory_consumption(prompt: str,
                                                            n_tokens_to_generate: int):
    tokenizer = GPT2BPETokenizer()
    indices = tokenizer.encode(prompt)

    print('Prompt:', prompt)
    print('Tokens to generate:', n_tokens_to_generate)
    model_param_fp32s = sum(p.numel() for p in model.parameters())
    print('Model Param Memory Consumption:', f'{model_param_fp32s / 1e6 * 4:.1f} MB')
    print()

    for use_kv_cache in [False, True]:
        if use_kv_cache:
            print('USING KV CACHE')
            print('==============')
        else:
            print('NOT USING KV CACHE')
            print('==================')

        model.clear_kv_cache()
        start_time = time.time()

        generator = gutils.generate_text(model, tokenizer, prompt,
                                         n_tokens_to_gen=n_tokens_to_generate,
                                         top_k=100,
                                         top_p=0.95,
                                         sample=True,
                                         print_stream=False,
                                         use_kv_cache=use_kv_cache)

        response = ''.join(list(generator))
        end_time = time.time()

        if use_kv_cache:
            kv_cache_fp32s = model.decoder_blocks[0].attn.kv_cache[0].numel()
            peak_kv_cache_fp32s = (len(indices) + n_tokens_to_generate) * kv_cache_fp32s // len(indices)
        else:
            peak_kv_cache_fp32s = 0

        print('Generation time:'.ljust(28), f'{end_time - start_time:.1f} sec')
        print('KV Cache Memory Consumption:'.ljust(28), f'{peak_kv_cache_fp32s / 1e6 * 4:.1f} MB')
        print()

    print('Response:', response.lstrip())

In [11]:
import os
import random
import numpy as np

def set_all_seeds(seed: int):
    # Python/hash seed
    os.environ['PYTHONHASHSEED'] = str(seed)
    # cuBLAS deterministic workspace (NVIDIA recommended for determinism)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

    # stdlib / numpy
    random.seed(seed)
    np.random.seed(seed)

    # torch (torch already imported elsewhere in the notebook)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    print(f"Deterministic seeds and flags set to: {seed}")

# call with the seed you used elsewhere (e.g. 1337)
set_all_seeds(1337)

Deterministic seeds and flags set to: 1337


In [12]:
profile_kv_cache_generation_time_and_memory_consumption(
    'Note: KV cache memory figures are only on single batch and a relatively small context length, '
    'and in practice will be much larger.',
    n_tokens_to_generate = 100
)

Prompt: Note: KV cache memory figures are only on single batch and a relatively small context length, and in practice will be much larger.
Tokens to generate: 100
Model Param Memory Consumption: 497.8 MB

NOT USING KV CACHE
Generation time:             3.7 sec
KV Cache Memory Consumption: 0.0 MB

USING KV CACHE
Generation time:             1.5 sec
KV Cache Memory Consumption: 1.8 MB

Response: KV cache size is for simple instances of SQLite; all we need is to make sure that the page size is fixed. For instance, I should see 0 MB in my DLL on 1-2 SQLite 4.8 (12.2 GiB in total), I should have 1 MB in my main (non-active) page and 3 MB in the pages in the .sqlite databases (all pages in the DLL should be active).

When I find


# ShakeSpeare

In [10]:
import os
import torch

In [11]:
block_size = 256
n_layers = 6
n_heads = 6
embed_dim = 384
dropout_p = 0.2
bias=False
device = "cuda" if torch.cuda.is_available() else "cpu"

In [12]:
model_args = dict(n_layers=n_layers, n_heads=n_heads, embed_dim=embed_dim, block_size=block_size,
                  bias=bias, vocab_size=50304, dropout_p=dropout_p) # start with model_args from command line

In [13]:
model = GPT(**model_args)

In [14]:
ckpt_path = os.path.join("/home/nyxx/my_project/marejv2/out-shakespeare-char", 'ckpt.pt')
checkpoint = torch.load(ckpt_path, map_location=device)
state_dict = checkpoint['model']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
model.load_state_dict(state_dict)

<All keys matched successfully>

In [15]:
model.eval()
model.to(device)

GPT(
  (dropout): Dropout(p=0.2, inplace=False)
  (word_embeddings): Embedding(50304, 384)
  (position_embeddings): Embedding(256, 384)
  (decoder_blocks): ModuleList(
    (0-5): 6 x DecoderBlock(
      (dropout): Dropout(p=0.2, inplace=False)
      (ln1): LayerNorm()
      (attn): MultiheadAttention(embed_dim=384, n_heads=6)
      (ln2): LayerNorm()
      (ffn): FeedForwardBlock(
        (linear1): Linear(in_features=384, out_features=1536, bias=True)
        (linear2): Linear(in_features=1536, out_features=384, bias=True)
      )
    )
  )
  (layer_norm): LayerNorm()
)

In [23]:
def profile_kv_cache_generation_time_and_memory_consumption(prompt: str,
                                                            n_tokens_to_generate: int):
    tokenizer = GPT2BPETokenizer()
    indices = tokenizer.encode(prompt)

    print('Prompt:', prompt)
    print('Tokens to generate:', n_tokens_to_generate)
    model_param_fp32s = sum(p.numel() for p in model.parameters())
    print('Model Param Memory Consumption:', f'{model_param_fp32s / 1e6 * 4:.1f} MB')
    print()

    for use_kv_cache in [False, True]:
        if use_kv_cache:
            print('USING KV CACHE')
            print('==============')
        else:
            print('NOT USING KV CACHE')
            print('==================')

        model.clear_kv_cache()
        start_time = time.time()

        generator = gutils.generate_text(model, tokenizer, prompt,
                                         n_tokens_to_gen=n_tokens_to_generate,
                                         top_k=100,
                                         top_p=0.95,
                                         sample=True,
                                         print_stream=False,
                                         use_kv_cache=use_kv_cache)

        response = ''.join(list(generator))
        end_time = time.time()

        if use_kv_cache:
            kv_cache_fp32s = model.decoder_blocks[0].attn.kv_cache[0].numel()
            peak_kv_cache_fp32s = (len(indices) + n_tokens_to_generate) * kv_cache_fp32s // len(indices)
        else:
            peak_kv_cache_fp32s = 0

        print('Generation time:'.ljust(28), f'{end_time - start_time:.1f} sec')
        print('KV Cache Memory Consumption:'.ljust(28), f'{peak_kv_cache_fp32s / 1e6 * 4:.1f} MB')
        print()

    print('Response:', response.lstrip())

In [24]:
profile_kv_cache_generation_time_and_memory_consumption(
    'Note: KV cache memory figures are only on single batch and a relatively small context length, '
    'and in practice will be much larger.',
    n_tokens_to_generate = 220
)

Loading file from cache: /home/nyxx/.cache/candle/gpt2_encoder.json
Loading file from cache: /home/nyxx/.cache/candle/gpt2_vocab.bpe
Prompt: Note: KV cache memory figures are only on single batch and a relatively small context length, and in practice will be much larger.
Tokens to generate: 220
Model Param Memory Consumption: 120.3 MB

NOT USING KV CACHE
Generation time:             0.7 sec
KV Cache Memory Consumption: 0.0 MB

USING KV CACHE
Generation time:             0.7 sec
KV Cache Memory Consumption: 3.5 MB

Response: +!4YPT'"SL["[OLT"I`"[OL"YLKLLT"OH]L"H["[OLL'!;V["[OLPY"OLH]LUZ"VM"OPZ"[OYVUL)!!0.=B92A+!DOLYLMVYL'"ZPY'"OV^"ZOL"JVUZ\S-!!@LJVUK"0P[PaLU+!;V'"UV"TVYL)!!0.=B92A+!;V"ZOHSS"[OLL'"MVY"T`"MH[OLY&Z"Z[HUKZ)!!3PYZ["0P[PaLU+!AOL"]


In [ ]:
def profile_kv_cache_generation_time_and_memory_consumption(prompt: str,
                                                            n_tokens_to_generate: int):
    tokenizer = GPT2BPETokenizer()
    indices = tokenizer.encode(prompt)

    print('Prompt:', prompt)
    print('Tokens to generate:', n_tokens_to_generate)
    model_param_fp32s = sum(p.numel() for p in model_ss.parameters())
    print('Model Param Memory Consumption:', f'{model_param_fp32s / 1e6 * 4:.1f} MB')
    print()

    for use_kv_cache in [False, True]:
        if use_kv_cache:
            print('USING KV CACHE')
            print('==============')
        else:
            print('NOT USING KV CACHE')
            print('==================')

        model_ss.clear_kv_cache()
        start_time = time.time()

        generator = gutils.generate_text(model_ss, tokenizer, prompt,
                                         n_tokens_to_gen=n_tokens_to_generate,
                                         top_k=100,
                                         top_p=0.95,
                                         sample=True,
                                         print_stream=False,
                                         use_kv_cache=use_kv_cache)

        response = ''.join(list(generator))
        end_time = time.time()

        if use_kv_cache:
            kv_cache_fp32s = model_ss.decoder_blocks[0].attn.kv_cache[0].numel()
            peak_kv_cache_fp32s = (len(indices) + n_tokens_to_generate) * kv_cache_fp32s // len(indices)
        else:
            peak_kv_cache_fp32s = 0

        print('Generation time:'.ljust(28), f'{end_time - start_time:.1f} sec')
        print('KV Cache Memory Consumption:'.ljust(28), f'{peak_kv_cache_fp32s / 1e6 * 4:.1f} MB')
        print()

    print('Response:', response.lstrip())

In [ ]:
profile_kv_cache_generation_time_and_memory_consumption(
    'Note: KV cache memory figures are only on single batch and a relatively small context length, '
    'and in practice will be much larger.',
    n_tokens_to_generate = 100
)

In [ ]:
import tiktoken

In [ ]:
enc = tiktoken.get_encoding("gpt2")
encode = lambda s: enc.encode(s, allowed_special={"<|endoftext|>"})
decode = lambda l: enc.decode(l)

In [ ]:
start_ids = encode("""
Characters in the Play
======================
Caius MARTIUS, later Caius Martius CORIOLANUS
VOLUMNIA, his mother
VIRGILIA, his wife
YOUNG MARTIUS, their son
                   """)
x = (torch.tensor(start_ids, dtype=torch.long, device=device)[None, ...])
for k in range(1):
    y = model_ss.generate(x, 5, temperature=0.8, top_k=95)
    print(decode(y[0].tolist()))
    print('---------------')

In [ ]:
!pwd

In [ ]:
"""
Sample from a trained model
"""
import os
import pickle
from contextlib import nullcontext
import torch
import tiktoken

# -----------------------------------------------------------------------------
init_from = 'resume' # either 'resume' (from an out_dir) or a gpt2 variant (e.g. 'gpt2-xl')
out_dir = '/home/nyxx/my_project/marejv2/out-shakespeare-char' # ignored if init_from is not 'resume'
start = "Hello" # or "<|endoftext|>" or etc. Can also specify a file, use as: "FILE:prompt.txt"
num_samples = 10 # number of samples to draw
max_new_tokens = 500 # number of tokens generated in each sample
temperature = 0.8 # 1.0 = no change, < 1.0 = less random, > 1.0 = more random, in predictions
top_k = 200 # retain only the top_k most likely tokens, clamp others to have 0 probability
seed = 1337
device = 'cuda' # examples: 'cpu', 'cuda', 'cuda:0', 'cuda:1', etc.
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16' # 'float32' or 'bfloat16' or 'float16'
compile = False # use PyTorch 2.0 to compile the model to be faster
# exec(open('configurator.py').read()) # overrides from command line or config file
# -----------------------------------------------------------------------------

torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cuda.matmul.allow_tf32 = True # allow tf32 on matmul
torch.backends.cudnn.allow_tf32 = True # allow tf32 on cudnn
device_type = 'cuda' if 'cuda' in device else 'cpu' # for later use in torch.autocast
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

# model
if init_from == 'resume':
    # init from a model saved in a specific directory
    ckpt_path = os.path.join(out_dir, 'ckpt.pt')
    checkpoint = torch.load(ckpt_path, map_location=device)
    # gptconf = GPTConfig(**checkpoint['model_args'])
    model = GPT(**model_args)
    state_dict = checkpoint['model']
    unwanted_prefix = '_orig_mod.'
    for k,v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    model.load_state_dict(state_dict)
elif init_from.startswith('gpt2'):
    # init from a given GPT-2 model
    model = GPT.from_pretrained(init_from, dict(dropout=0.0))

model.eval()
model.to(device)
if compile:
    model = torch.compile(model) # requires PyTorch 2.0 (optional)

# look for the meta pickle in case it is available in the dataset folder
load_meta = False
if init_from == 'resume' and 'config' in checkpoint and 'dataset' in checkpoint['config']: # older checkpoints might not have these...
    meta_path = os.path.join('data', checkpoint['config']['dataset'], 'meta.pkl')
    load_meta = os.path.exists(meta_path)
if load_meta:
    print(f"Loading meta from {meta_path}...")
    with open(meta_path, 'rb') as f:
        meta = pickle.load(f)
    # TODO want to make this more general to arbitrary encoder/decoder schemes
    stoi, itos = meta['stoi'], meta['itos']
    encode = lambda s: [stoi[c] for c in s]
    decode = lambda l: ''.join([itos[i] for i in l])
else:
    # ok let's assume gpt-2 encodings by default
    print("No meta.pkl found, assuming GPT-2 encodings...")
    enc = tiktoken.get_encoding("gpt2")
    encode = lambda s: enc.encode(s, allowed_special={"<|endoftext|>"})
    decode = lambda l: enc.decode(l)

# encode the beginning of the prompt
if start.startswith('FILE:'):
    with open(start[5:], 'r', encoding='utf-8') as f:
        start = f.read()
start_ids = encode(start)
x = (torch.tensor(start_ids, dtype=torch.long, device=device)[None, ...])

# run generation
with torch.no_grad():
    for k in range(num_samples):
        y = model.generate_sample(x, max_new_tokens, temperature=temperature, top_k=top_k)
        print(decode(y[0].tolist()))
        print('---------------')

In [ ]:
model.generate_sample

In [ ]:
model.position_embeddings.weight

In [ ]:
state_dict["position_embeddings.weight"]

In [ ]:
state_dict.keys()